<a href="https://colab.research.google.com/github/GurneeshBudhiraja/ArangoDB-Hackathon/blob/main/ArangoDB_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ArangoDB Hackathon :: Synthea_P100 Dataset Agent

In [1]:
# Required packages
!pip install nx-arangodb
!pip install arango-datasets
!nvidia-smi
!nvcc --version
!pip install nx-cugraph-cu12 --extra-index-url https://pypi.nvidia.com
!pip install google-genai

/bin/bash: line 1: nvidia-smi: command not found
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [56]:
# Import required modules
import json
import pprint
from google.colab import userdata
from pydantic import BaseModel
from enum import Enum
from typing import Optional, List


import networkx as nx
import nx_arangodb as nxadb

from arango import ArangoClient
from arango_datasets import Datasets


# Gemini SDK Packages
from google import genai

In [57]:
# Imports the secrets from the Colab notebook
ARANGO_HOST = userdata.get("ARANGO_HOST")
ARANGO_PASSWORD = userdata.get("ARANGO_PASSWORD")
ARANGO_USERNAME = userdata.get("ARANGO_USERNAME")
GEMINI_API = userdata.get("GEMINI_API_KEY")

# Gemini models
GEMINI_FLASH_MODEL = "gemini-2.0-flash"
GEMINI_FLASH_LITE_MODEL = "gemini-2.0-flash-lite"
GEMINI_PRO_MODEL = "gemini-1.5-pro"


# Creates a db connection
arango_client = ArangoClient(hosts=ARANGO_HOST).db(username=ARANGO_USERNAME, password=ARANGO_PASSWORD, verify=True)

# Creates the genai instance
gemini_client = genai.Client(
    api_key=GEMINI_API
)

In [58]:
# Creates the dataset connection using the db object
datasets = Datasets(arango_client)

DATASET_NAME = "SYNTHEA_P100"

# Conditionally Loads the Synthea P100 dataset in Arango
if not arango_client.has_graph(DATASET_NAME):
  datasets.load(dataset_name=DATASET_NAME)
else:
  print(f"{DATASET_NAME} is already in ArangoDB.")

Output()

SYNTHEA_P100 is already in ArangoDB.


In [59]:
# Connects with the Graph in ArangoDB
graph = None
if arango_client.has_graph(DATASET_NAME):
  graph = nxadb.Graph(name=DATASET_NAME, db=arango_client)
else:
  print("Graph does not exist in Arango DB")

print(graph)

[11:10:47 +0000] [INFO]: Graph 'SYNTHEA_P100' exists.
INFO:nx_arangodb:Graph 'SYNTHEA_P100' exists.
[11:10:47 +0000] [INFO]: Default node type set to 'allergies'
INFO:nx_arangodb:Default node type set to 'allergies'


Graph named 'SYNTHEA_P100' with 145514 nodes and 311701 edges


## Dictionary of System Prompts and schemas

In [60]:
prompts_dict = {
    #  System prompt for the starter function
    "starter_prompt":"""
        Your main job is to find the relevance of the prompt or suggest the proper agent for solving the user_query.
        For finding the relevance, you are given with the Synthea_P100 dataset in ArangoDB. If the initial user query is not related to inquiring the information about this dataset, they you would deny the request and will not entertain it further. For instance, the irrelevant user_queries that are beyond your scope is but not limited to, What is your name?, What is the weather outside?,  What is the capital of Canada? or anything that is not related to the Synthea_P100 dataset. The examples of relevant queries are: What is the name of the patient whose patient id is patients/xxxx-yyyy-zzzz, How many patients have allergies to Latex and groundnuts, and anything that is related to the Synthea_P100 dataset. The data is stored in the form of graph and collections in ArangoDB and here are the Vertices/Nodes and Edges in the Synthea_P100 dataset:
            Vertex Collections:
                allergies
                careplans
                conditions
                devices
                encounters
                imaging_studies
                immunizations
                medications
                observations
                organizations
                patients
                payers
                procedures
                providers
                supplies

            Edge Collections:
                encounters_to_allergies
                encounters_to_careplans
                encounters_to_conditions
                encounters_to_devices
                encounters_to_imaging_studies
                encounters_to_immunizations
                encounters_to_medications
                encounters_to_observations
                encounters_to_procedures
                encounters_to_supplies
                organizations_to_encounters
                organizations_to_providers
                patients_to_allergies
                patients_to_careplans
                patients_to_conditions
                patients_to_devices
                patients_to_encounters
                patients_to_imaging_studies
                patients_to_immunizations
                patients_to_medications
                patients_to_observations
                patients_to_procedures
                patients_to_supplies
                payers_to_encounters
                payers_to_medications
                providers_to_encounters

        If anything that is asked can not be found even in the Synthea_P100 dataset, in that cases too you will deny the request.

        Another thing that you have to do is, find the right agent for solving the user queries. So, there are 3 agents named as aql, networkX and hybrid. Analyze whether it is best to solve the user query using the AQL, NetworkX algorithm or hybrid(involves using both AQL and NetworkX queries) and select the agent accordingly.
    """,
    #  System prompt for the check memory
    "check_memory_prompt" : "Go throught the memory list that will be provided and try to find the answer to the user question. It is alright that if the answer to the user question is not present. Make sure to properly check if the answer to the user question is present in the memory list that will be provided. Also, please make sure that if you do find the info in the memory, reply the user in the natural language that should not sound robotic and if you got a question that is not related to the Synthea_P100 dataset, deny that requests too positively. Please make sure if the same or another version of the given user query is mentioned in the memory, then only you would answer the question. If you do not have the same or another same version of the question in the memory in that cases you would not use the memory to answer the user question.",
    #  System prompt for the AQL agent
    "aql_agent_prompt":"""
        You are the smart agent whose main job is to use the required tools to help answer the user query when the data that is stored is Synthea_P100. I will also provide you the list that contains past asked user questions and answers made by you. The format of the history list would be a list containing the dictionary like below:
            {
                "user_query":str,
                "agent_response":str
            }
        If you think that the complete whole answer that the user wants is in the history, you can reply from there otherwise follow the below instructions.

        Please make sure to always run the chunker tool that would help break the complex user queries into simple natural language queries. The chunker tool would return list of queries that you have use the proper tool to convert the simpler queries into the AQL queries and then also execute those AQL queries using the assigned tool. You will only rely on the tools that are provided to you to maintain the correctness and working of the application running.

        At the end, for all the data that you collected from different tools during the opearation, combine that result into the natural language and present to the user. Make sure the final response should give an idea at the end what sort data has been fetched by you and what data is missing.

        Also, before running any tool check that whether the user query is valid and would require fetching the data from the Synthea_P100 dataset. If it is not, you can deny the request.
    """

}


In [61]:
# Output Schema
class AgentChoice(str, Enum):
    AQL = "aql"
    NETWORKX = "networkX"
    HYBRID = "hybrid"
    NULL = "null"

class StarterFunctionSchema(BaseModel):
    is_relevant: bool
    not_relevant_message: Optional[str]
    is_agent_recommended: bool
    agent_choice: AgentChoice


class CheckMemorySchema(BaseModel):
    is_present:bool
    memory_answer: Optional[str]

# AQL Agent and  AQL Agent's Tool Schema
class AQLSchema(BaseModel):
  query:str

class ChunkerSchema(BaseModel):
  queries: str

## Checks the user query's relevance and suggests the right agent accordingly

#### Initializes the memory array to store the chat messages and creates the basic llm



In [62]:
agent_memory = [
    {
        "user_query":"What is the full name of the patient whose patient id is patients/",
        "user_query_answer":"""{
            "BIRTHDATE": "1951-08-10",
            "SSN": "999-99-9040",
            "DRIVERS": "S99991675",
            "PASSPORT": "X19696800X",
            "PREFIX": "Mr.",
            "FIRST": "Frankie174",
            "LAST": "Schinner682",
            "MARITAL": "M",
            "RACE": "white",
            "ETHNICITY": "nonhispanic",
            "GENDER": "M",
            "BIRTHPLACE": "North Adams  Massachusetts  US",
            "ADDRESS": "989 Kunze Orchard Unit 36",
            "CITY": "Lynn",
            "STATE": "Massachusetts",
            "COUNTY": "Essex County",
            "FIPS": 25009,
            "ZIP": 1907,
            "LAT": 42.4402758250332,
            "LON": -70.95449283370179,
            "HEALTHCARE_EXPENSES": 276780.33,
            "HEALTHCARE_COVERAGE": 568451.33,
            "INCOME": 124300
        }"""
    }
]

#### Function to check user query's relevance, uses memory to answer the repeated questions, and suggests the right agent

In [63]:

def starter_function(user_query:str):
    json_starter_reponse = gemini_client.models.generate_content(
        model=GEMINI_FLASH_LITE_MODEL,
        config = {
            "system_instruction": prompts_dict["starter_prompt"],
            "response_mime_type": "application/json",
            "response_schema":StarterFunctionSchema
        },
        contents=[f"user_query: {user_query}. In the final answer do not mention about the sources, just answer the user in the form of natural language."]
    )
    starter_response = json.loads(json_starter_reponse.text)
    print(f"================= Currently at : : Starter Function =====================\n")
    return starter_response


def check_memory(user_query:str):
    json_check_memory_response = gemini_client.models.generate_content(
        model=GEMINI_FLASH_MODEL,
        config = {
            "system_instruction": prompts_dict["check_memory_prompt"],
            "response_mime_type": "application/json",
            "response_schema":CheckMemorySchema
        },
        contents=f"user_query: {user_query} and the memory list is {agent_memory}"
    )
    print(f"================= Currently at : : Check Memory Function =====================\n")
    check_memory_response = json.loads(json_check_memory_response.text)
    return check_memory_response



## AQL Agent






##### AQL Agent Specific Prompts

In [64]:
aql_tool_system_prompts = {
    "aql_generator": """
        You are the ai assistant who will generate AQL queries. The ArangoDB has Synthea_P100 dataset with the following vertex and edge collections:

        Vertex Collections:
            allergies
            careplans
            conditions
            devices
            encounters
            imaging_studies
            immunizations
            medications
            observations
            organizations
            patients
            payers
            procedures
            providers
            supplies

        Edge Collections:
            encounters_to_allergies
            encounters_to_careplans
            encounters_to_conditions
            encounters_to_devices
            encounters_to_imaging_studies
            encounters_to_immunizations
            encounters_to_medications
            encounters_to_observations
            encounters_to_procedures
            encounters_to_supplies
            organizations_to_encounters
            organizations_to_providers
            patients_to_allergies
            patients_to_careplans
            patients_to_conditions
            patients_to_devices
            patients_to_encounters
            patients_to_imaging_studies
            patients_to_immunizations
            patients_to_medications
            patients_to_observations
            patients_to_procedures
            patients_to_supplies
            payers_to_encounters
            payers_to_medications
            providers_to_encounters

        Your job is to generate an AQL query based on the user's query. Make sure the query is correct and achieves the purpose and gets the info that is needed. Make sure the AQL query should work on the Synthea_P100 dataset and make sure that the collection names, field names, and any other thing you mention is compatible with the dataset that is in the ArangoDB dataset.
  """,
    "chunker_prompt":"""
        Your job is to take in the user query and break into the list of simple queries for easy retrieval. For instance, if the initial user queries ask about so many data from the db break those queries into simple queries with proper data so that at the end we can get collective data for the user queries for the AI model to use this to answer the user's initial query.
        Note: You will be dividing the chunks for the Synthea_P100 dataset stored in the ArangoDb. Your only job is to break the query into the list of simple queries.

        Keep in mind your job is not to create any queries but rather conver the user query into simple natural queries as if you are the human who is modifying the complex natural language queries into simple natural language queries for the further tools to act upon.
    """,

}

#### AQL Agent Specific Tools

In [65]:
# Breaks the complex user query into smaller simpler queries(questions)
def chunker(user_query:str)->list[str]:
  """
  The tool is used to divide the user query into simple chunks(queries) for further tools

  Args:
  user_query: The initial user_query entered by the user

  Returns:
  The list of strings that contains queries for further tools are sent
  """
  print("================= USING THE chunker TOOL =================")
  json_repsonse = gemini_client.models.generate_content(
      model=GEMINI_FLASH_MODEL,
      config={
          "system_instruction": aql_tool_system_prompts["chunker_prompt"],
          "response_mime_type": "application/json",
          "response_schema":list[str]
      },
      contents=[user_query]
  )
  response = json.loads(json_repsonse.text)
  print(f"================= The new query is =================\n{response}")
  return response


# Generates aql queries
def aql_generator(user_query: str) -> dict[str, str]:
  """
  This is used to generate the AQL queries using the user query

  Args:
    user_query: The query asked by the user

  Returns:
    Returns a dict with a key 'query' and AQL query which will be a string
  """
  print("================= USING THE aql_generator TOOL =================")
  response = gemini_client.models.generate_content(
      model=GEMINI_PRO_MODEL,
      config={
          "system_instruction": aql_tool_system_prompts["aql_generator"],
          "response_mime_type": "application/json",
          "response_schema":AQLSchema
      },
      contents=[user_query]
  )
  return (json.loads(response.text))


def aql_executor(aql_query:str):
  """
  This is used to executed the queries that are in the AQL format using the arangoDB client

  Args:
    aql_query: This is the AQL query that can be executed in the ArangoDB

  Returns:
    This returns the list of the data from the ArangoDB. The empty list means that nothing related to that query is available
  """
  print("================= USING THE AQL_executor TOOL =================")
  print(f"AQL Query:\n{aql_query}")
  cursor = arango_client.aql.execute(aql_query)
  return list(cursor)



In [66]:
def aql_agent(user_query:str):
    print("====================== aql_agent ====================== \n\n")

    # Agent tools
    tools = [aql_generator, aql_executor,chunker]

    # Agent config
    config = {
        'tools': tools,
        'system_instruction': prompts_dict["aql_agent_prompt"],
    }

    try:
        agent_response = gemini_client.models.generate_content(
        model=GEMINI_FLASH_MODEL,
        config=config,
        contents=[user_query]
        )
        return agent_response.text

    except Exception as e:
        print("EXCEPTION HAS OCCURED IN aql_agent :: Retrying again")
        print(e)

        # aql_agent(user_query)

## Takes the user input, checks the query, and suggest proper method for the answer to the user query

In [69]:
# Runs until a user query has been mentioned by the user
agent_memory = []
while True:
    user_query = input("Enter your query ").strip()
    if not user_query:
        break
    else:
        starter_response = starter_function(user_query=user_query)
        if not starter_response["is_agent_recommended"]:
            print(starter_response["not_relevant_message"], end="\n\n")
            continue
        elif (starter_response["is_relevant"]):
            match starter_response["agent_choice"]:
                case "aql":
                    print(f"This query will go through AQL as agent choice is: {starter_response['agent_choice']}")
                    aql_agent_reponse = aql_agent(user_query=user_query)
                    print(aql_agent_reponse)
                    # Updating the agent memory
                    continue
                case "networkX":
                    print(f"This query will go through networkX as agent choice is: {starter_response['agent_choice']}")
                    continue
                case "hybrid":
                    print(f"This query will go through hybrid method as agent choice is: {starter_response['agent_choice']}")
                    continue
                case _:
                    print(f"Invalid agent choice: {starter_response}")
                    continue
        else:
            print("Unexpected: ", starter_response)


Enter your query get me the full details of the patient whose patient id is patients/01fd0320-1260-3613-95fb-7703f53e6a08 and also get me the immunization details about this patient
================= Currently at : : Starter Function =====================

This query will go through hybrid method as agent choice is: hybrid
Enter your query get me the full details of the patient whose patient id is patients/01fd0320-1260-3613-95fb-7703f53e6a08 and also get me the immunization details about this patient using the AQL
================= Currently at : : Starter Function =====================

This query will go through AQL as agent choice is: aql
====================== aql_agent ====================== 


================= USING THE chunker TOOL =================
================= The new query is =================
['Get the full details of the patient with patient ID patients/01fd0320-1260-3613-95fb-7703f53e6a08.', 'Retrieve immunization details for the patient with patient ID patients/01f

KeyboardInterrupt: Interrupted by user